# NQCT SDK walkthrough — submit OpenQASM

End-to-end notebook for the **direct QASM** flow against a live NQCT Cloud stack:

1. Authenticate and discover backends
2. Submit inline OpenQASM 3 via `client.submit_job()` → `POST /jobs`
3. Monitor the async **Job** with `job.wait()` and fetch results
4. List recent jobs

**Prerequisites**

1. Create an API key in the web UI profile page ([cloud.nqct.org](https://cloud.nqct.org))
2. Link at least one **online** backend in the admin UI
3. Install notebook deps: `uv sync --extra notebook` (or `pip install -e ".[notebook]"`)

The client defaults to production (`https://api.nqct.org/api/v1`). For local `nqct start`, pass `url="http://localhost:8000/api/v1"` to `save_account` / `NQCTClient`.

Never commit API keys.


In [ ]:
from __future__ import annotations

from pprint import pprint

from nqct import NQCTClient
from nqct.exceptions import NQCTError

# Replace with your API key from the NQCT Cloud profile page
API_KEY = "YOUR_API_KEY_HERE"

# One-time: write ~/.nqct/credentials.json (mode 0600).
# URL defaults to https://api.nqct.org/api/v1 — pass url= for local nqct start.
NQCTClient.save_account(api_key=API_KEY)
print("Saved account to ~/.nqct/credentials.json")


## Load saved account

After `save_account`, construct the client with no arguments — it reads `~/.nqct/credentials.json` (or `NQCT_API_KEY` / `NQCT_URL` from the environment).


In [ ]:
client = NQCTClient()
print(f"API URL: {client.url}")
client


## Authentication — `GET /auth/me`

In [ ]:
profile = client.me()
pprint({k: profile.get(k) for k in ("id", "username", "email", "full_name", "is_superuser")})

## Backends — `GET /backends`

List catalog backends, inspect one online simulator, and pick the least-busy target for the QASM job below.

In [ ]:
backends = client.backends(limit=20)
print(f"Found {len(backends)} backend(s)")
for b in backends:
    print(f"  {b.id:30} {b.name:25} {b.type:10} {b.status}")

In [ ]:
online = client.backends(status="online", type="simulator")
if not online:
    print("No online simulators — link a backend in the NQCT admin UI.")
else:
    backend = online[0]
    detail = client.backend(backend.id)
    print(f"Detail: {detail.name} ({detail.id}), qubits={detail.qubits}")

    queue = detail.queue_status()
    print(
        f"Queue depth={queue.queue_depth}, "
        f"queued={queue.queued}, running={queue.running}, "
        f"est. wait={queue.estimated_wait_time_seconds:.1f}s"
    )

In [ ]:
try:
    picked = client.least_busy(type="simulator")
    print(f"Least busy simulator: {picked.name} ({picked.id})")
except LookupError as exc:
    print(exc)

## Submit OpenQASM — `POST /jobs`

Enqueue a Bell-state circuit on a managed backend. Leave `BACKEND_ID` as `None` to pick the least-busy simulator, or set it to pin a backend (e.g. `"qiskit-aer-local"`).

Use `source="api"` so the Jobs UI can distinguish SDK submits from UI `direct_qasm` jobs.


In [ ]:
BELL_QASM = """OPENQASM 3.0;
include "stdgates.inc";
qubit[2] q;
bit[2] c;
h q[0];
cx q[0], q[1];
c = measure q;
"""

BACKEND_ID: str | None = None  # e.g. "qiskit-aer-local"; None → least_busy simulator
SHOTS = 1024

backend = client.backend(BACKEND_ID) if BACKEND_ID else client.least_busy(type="simulator")
print(f"Backend: {backend.name} ({backend.id})")

job = client.submit_job(
    qasm=BELL_QASM,
    backend_id=backend.id,
    shots=SHOTS,
    source="api",
    metadata={"label": "sdk-walkthrough-bell"},
)
print(f"Queued job {job.id}  status={job.status}  queue_position={job.queue_position}")


## Monitor job — `job.wait()`

Poll until the job reaches a terminal status (`done` / `failed` / `cancelled`), then print results or error logs.


In [ ]:
finished = job.wait(timeout=600, interval=5)
print(f"Finished: status={finished.status}  execution_time={finished.execution_time_seconds}")

if finished.status == "done":
    results = finished.result()
    pprint(results)
    counts = results.get("counts") if isinstance(results, dict) else None
    if counts:
        print("Counts:")
        for bitstring, n in sorted(counts.items(), key=lambda kv: (-kv[1], kv[0])):
            print(f"  {bitstring}: {n}")
else:
    if finished.error_message:
        print(f"error_message: {finished.error_message}")
    try:
        for line in finished.logs()[:30]:
            print(line)
    except NQCTError as exc:
        print(f"Could not fetch logs: {exc.message}")


## List jobs — `GET /jobs`

Filter by `source="api"` to see SDK submits from this walkthrough.


In [ ]:
all_jobs = client.jobs(limit=10)
print(f"Recent jobs (any source): {len(all_jobs)}")
for j in all_jobs:
    print(
        f"  {j.id}  source={j.source or '-':15} {j.status:10} "
        f"shots={j.shots} backend={j.backend_id}"
    )

api_jobs = client.jobs(source="api", limit=5)
print(f"\nRecent API-source jobs: {len(api_jobs)}")
for j in api_jobs:
    print(f"  {j.id}  {j.status:10} shots={j.shots} backend={j.backend_id}")


## Cleanup


In [ ]:
client.close()
print("Client closed.")
